# HW02 — MLflow Experiment Tracking

In HW01, you built a versioned feature dataset for the Airbnb listing availability problem.

In this notebook, you will train several model versions and track them in MLflow.

The goal is not only to get a high score. The goal is to make every experiment reproducible:

- which dataset version was used
- which features were used
- which model was trained
- which parameters were used
- which metrics were produced
- which artifacts were saved
- which run should be considered the final candidate

MLflow server:

```text
http://185.50.38.163:33014
```

Use your assigned MLflow username/password and your assigned experiment name from the credentials sheet.

## Required output

By the end of this notebook, you must have:

1. At least **5 MLflow runs**.
2. At least **3 different experiment types**:
   - one intentionally leaky run
   - one baseline run
   - at least one clean real model
3. Logged parameters, metrics, tags, artifacts, and an sklearn Pipeline model.
4. A run comparison table.
5. One selected final candidate run.
6. A short explanation of why that run was selected.

Do not use future/label columns in your final clean model.

In [3]:
# If needed, install these in your local environment first:
# pip install pandas numpy scikit-learn matplotlib mlflow pyarrow

import os
import json
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
)

import mlflow
import mlflow.sklearn

RANDOM_STATE = 42

## 1. Configure MLflow

Fill in your assigned MLflow credentials.

Important:

- `MLFLOW_TRACKING_URI` is the shared MLflow server.
- `MLFLOW_USERNAME` and `MLFLOW_PASSWORD` are **not** your database credentials.
- `EXPERIMENT_NAME` must be your own assigned experiment name, for example:

```text
qbc12_hw02_student_nazanin_hesari
```

Do not use someone else's experiment name.

In [4]:
MLFLOW_TRACKING_URI = "http://185.50.38.163:33014"

# TODO: replace these with your assigned MLflow credentials.
MLFLOW_USERNAME = os.getenv("MLFLOW_TRACKING_USERNAME", "student_your_username")
MLFLOW_PASSWORD = os.getenv("MLFLOW_TRACKING_PASSWORD", "your_mlflow_password")
EXPERIMENT_NAME = os.getenv("MLFLOW_EXPERIMENT_NAME", f"qbc12_hw02_{MLFLOW_USERNAME}")

if MLFLOW_USERNAME == "student_your_username" or MLFLOW_PASSWORD == "your_mlflow_password":
    raise ValueError("Replace MLFLOW_USERNAME, MLFLOW_PASSWORD, and EXPERIMENT_NAME with your assigned values.")

os.environ["MLFLOW_TRACKING_USERNAME"] = MLFLOW_USERNAME
os.environ["MLFLOW_TRACKING_PASSWORD"] = MLFLOW_PASSWORD

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

client = mlflow.tracking.MlflowClient()
experiment = client.get_experiment_by_name(EXPERIMENT_NAME)

print("MLflow tracking URI:", mlflow.get_tracking_uri())
print("Experiment:", experiment.name if experiment else None)
print("Experiment ID:", experiment.experiment_id if experiment else None)


MLflow tracking URI: http://185.50.38.163:33014
Experiment: qbc12_hw02_student_nima_hosseini
Experiment ID: 43


## 2. Load the HW01 dataset

Use the cleaned dataset produced by HW01.

Expected files:

```text
data/features/listing_availability_features_v1_audit_cleaned.csv
data/features/listing_availability_features_v1_audit_cleaned.parquet
data/features/listing_availability_features_v1_audit_cleaned_metadata.json
```

You may use CSV or Parquet. Parquet is preferred if available.

In [22]:
DATASET_VERSION = "v1_audit_cleaned"

FEATURE_DIR = Path("data/features")

parquet_path = FEATURE_DIR / f"listing_availability_features_{DATASET_VERSION}.parquet"
csv_path = FEATURE_DIR / f"listing_availability_features_{DATASET_VERSION}.csv"
metadata_path = FEATURE_DIR / f"listing_availability_features_{DATASET_VERSION}_metadata.json"

# TODO: load the dataset.
# Prefer Parquet if it exists, otherwise use CSV.
data_path = None

if parquet_path.is_file():
    data_path = parquet_path
elif csv_path.is_file():
    data_path = csv_path

if data_path is None:
    raise FileNotFoundError(
        f"Feature dataset not found. Expected {parquet_path} or {csv_path}. Run notebook 01 first."
    )

feature_df = (
    pd.read_parquet(data_path)
    if data_path.suffix == ".parquet"
    else pd.read_csv(data_path)
)

# TODO: load metadata if metadata_path exists.
metadata = {}

if metadata_path.is_file():
    with metadata_path.open("r", encoding="utf-8") as file:
        metadata = json.load(file)

print(feature_df.shape)
feature_df.head()


(10480, 33)


,listing_id,room_type,property_type,neighbourhood_name,accommodates,bedrooms,beds,bathrooms,listing_price,minimum_nights,...,available_days_last_30d,available_rate_last_30d,avg_minimum_nights_calendar_last_30d,avg_maximum_nights_calendar_last_30d,future_calendar_days_observed_30d,future_available_days_30d,future_available_rate_30d,high_demand_proxy,cutoff_date,dataset_version
0,27886,Private room,Private room in houseboat,Centrum-West,2,1.0,1.0,1.5,132.0,3,...,0,0.000000,3.0,30.0,30,0,0.0,1,2026-08-11,v1_audit_cleaned
1,28871,Private room,Private room in rental unit,Centrum-West,2,1.0,1.0,1.0,89.0,2,...,14,0.466667,2.0,730.0,30,21,0.7,0,2026-08-11,v1_audit_cleaned
2,29051,Private room,Private room in condo,Centrum-Oost,2,1.0,1.0,1.0,61.0,2,...,16,0.533333,2.0,730.0,30,0,0.0,1,2026-08-11,v1_audit_cleaned
3,44391,Entire home/apt,Entire rental unit,Centrum-Oost,4,2.0,NaN,1.5,NaN,3,...,0,0.000000,3.0,730.0,30,0,0.0,1,2026-08-11,v1_audit_cleaned
4,48373,Entire home/apt,Entire rental unit,Buitenveldert - Zuidas,4,2.0,NaN,1.5,NaN,3,...,0,0.000000,3.0,1125.0,30,0,0.0,1,2026-08-11,v1_audit_cleaned


## 3. Define target and forbidden columns

The target is:

```text
high_demand_proxy
```

The following columns must **not** be used as clean model inputs:

```text
listing_id
cutoff_date
dataset_version
future_calendar_days_observed_30d
future_available_days_30d
future_available_rate_30d
high_demand_proxy
```

Why?

- `high_demand_proxy` is the label.
- `future_*` columns are from the label window.
- `listing_id`, `cutoff_date`, and `dataset_version` are audit/entity fields, not predictive features.

You will intentionally use one future column in the **leaky run only** to show what leakage looks like. Your final model must be clean.

In [23]:
TARGET_COL = "high_demand_proxy"

FORBIDDEN_MODEL_COLUMNS = [
    "listing_id",
    "cutoff_date",
    "dataset_version",
    "future_calendar_days_observed_30d",
    "future_available_days_30d",
    "future_available_rate_30d",
    "high_demand_proxy",
]

# TODO: check that TARGET_COL exists.
if TARGET_COL not in feature_df.columns:
    raise KeyError(f"Missing target column: {TARGET_COL}")

# TODO: create y.
y = feature_df.loc[:, TARGET_COL].astype("int64")

# TODO: create clean feature list by excluding FORBIDDEN_MODEL_COLUMNS.
forbidden_cols = set(FORBIDDEN_MODEL_COLUMNS)
clean_feature_cols = [
    column
    for column in feature_df.columns
    if column not in forbidden_cols
]

# TODO: create X_clean.
X_clean = feature_df.loc[:, clean_feature_cols].copy()

assert not set(FORBIDDEN_MODEL_COLUMNS).intersection(X_clean.columns)
assert len(clean_feature_cols) > 0

print("Target distribution:")
print(y.value_counts(normalize=True).sort_index())

print("Clean feature count:", len(clean_feature_cols))
print(clean_feature_cols)


Target distribution:
high_demand_proxy
0    0.237214
1    0.762786
Name: proportion, dtype: float64
Clean feature count: 26
['room_type', 'property_type', 'neighbourhood_name', 'accommodates', 'bedrooms', 'beds', 'bathrooms', 'listing_price', 'minimum_nights', 'maximum_nights', 'instant_bookable', 'host_is_superhost', 'host_listing_count', 'total_reviews_before_cutoff', 'unique_reviewers_before_cutoff', 'avg_comment_len_before_cutoff', 'max_comment_len_before_cutoff', 'days_since_last_review', 'available_days_last_90d', 'available_rate_last_90d', 'avg_minimum_nights_calendar_last_90d', 'avg_maximum_nights_calendar_last_90d', 'available_days_last_30d', 'available_rate_last_30d', 'avg_minimum_nights_calendar_last_30d', 'avg_maximum_nights_calendar_last_30d']


## 4. Create one intentionally leaky feature set

This run is supposed to be wrong.

Create `X_leaky` by allowing `future_available_rate_30d` into the features.

The point is to show that a model can look excellent for the wrong reason. Log this run with:

```text
leakage_status = leaky
known_defect = uses future_available_rate_30d
```

Do not select this run as your final model.

In [24]:
LEAKAGE_COLUMN = "future_available_rate_30d"

# TODO: create leaky_feature_cols.
# It should include the clean features plus LEAKAGE_COLUMN.
# It must still exclude the target itself.
if LEAKAGE_COLUMN not in feature_df.columns:
    raise KeyError(f"Missing leakage demo column: {LEAKAGE_COLUMN}")

leaky_feature_cols = list(clean_feature_cols)

if LEAKAGE_COLUMN not in leaky_feature_cols:
    leaky_feature_cols.append(LEAKAGE_COLUMN)

leaky_feature_cols = [
    feature for feature in leaky_feature_cols
    if feature != TARGET_COL
]

X_leaky = feature_df.loc[:, leaky_feature_cols].copy()

print("Leaky feature count:", len(leaky_feature_cols))
print("Leakage column included:", LEAKAGE_COLUMN in leaky_feature_cols)


Leaky feature count: 27
Leakage column included: True


## 5. Train/test split

Use a stratified split.

Why stratified?

The target is not perfectly balanced, so the train and test sets should preserve the class ratio.

In [25]:
# TODO: split X_clean and y.
# Use test_size=0.20, random_state=42, stratify=y.

split_kwargs = {
    "test_size": 0.20,
    "random_state": 42,
    "stratify": y,
}

X_train, X_test, y_train, y_test = train_test_split(
    X_clean,
    y,
    **split_kwargs,
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Train target rate:", y_train.mean())
print("Test target rate:", y_test.mean())


Train shape: (8384, 26)
Test shape: (2096, 26)
Train target rate: 0.7627624045801527
Test target rate: 0.762881679389313


## 6. Build preprocessing

Use an sklearn `ColumnTransformer`.

Required preprocessing:

- numeric columns:
  - median imputation
  - standard scaling
- categorical columns:
  - most-frequent imputation
  - one-hot encoding

The logged model must be a full sklearn `Pipeline`, not just the estimator.

In [26]:
def make_one_hot_encoder():
    """Return OneHotEncoder compatible with multiple sklearn versions."""
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


# TODO: identify numeric_cols and categorical_cols from X_clean.
# Hint: numeric columns usually have dtype int/float.
# Everything else can be treated as categorical.

numeric_cols = X_clean.select_dtypes(
    include=["number", "bool", "boolean"]
).columns.to_list()

categorical_cols = X_clean.columns.difference(numeric_cols).tolist()

# raise NotImplementedError("Create numeric_cols and categorical_cols.")

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", make_one_hot_encoder()),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_transformer, numeric_cols),
        ("categorical", categorical_transformer, categorical_cols),
    ],
    remainder="drop",
)

print("Numeric columns:", len(numeric_cols))
print("Categorical columns:", len(categorical_cols))

Numeric columns: 23
Categorical columns: 3


## 7. Evaluation helpers

Complete the evaluation helper.

Every run must log the same metric set:

```text
accuracy
precision
recall
f1
roc_auc
```

Use `zero_division=0` for precision/recall/f1.

In [27]:
def get_positive_scores(model, X):
    """Return positive-class scores for binary classifiers."""
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, "decision_function"):
        raw = model.decision_function(X)
        return 1 / (1 + np.exp(-raw))
    return model.predict(X)


def evaluate_binary_classifier(model, X_test, y_test, threshold=0.5):
    """Evaluate a fitted binary classifier."""
    # TODO:
    # 1. get positive scores
    # 2. convert scores to predictions using threshold
    # 3. calculate accuracy, precision, recall, f1, roc_auc
    # 4. return metrics dict, y_pred, y_score

    y_score = get_positive_scores(model, X_test)

    y_pred = np.where(y_score >= threshold, 1, 0)

    metric_functions = {
        "test_accuracy": accuracy_score,
        "test_precision": lambda yt, yp: precision_score(yt, yp, zero_division=0),
        "test_recall": lambda yt, yp: recall_score(yt, yp, zero_division=0),
        "test_f1": lambda yt, yp: f1_score(yt, yp, zero_division=0),
    }

    metrics = {
        name: func(y_test, y_pred)
        for name, func in metric_functions.items()
    }

    metrics["test_roc_auc"] = roc_auc_score(y_test, y_score)
    metrics["prediction_threshold"] = float(threshold)


    return metrics, y_pred, y_score


## 8. Artifact helpers

Each serious run should save useful artifacts:

- confusion matrix image
- classification report JSON
- feature column list JSON
- dataset metadata snapshot JSON

Artifacts are important because MLflow should store more than scalar metrics.

In [28]:
ARTIFACT_DIR = Path("outputs/mlflow_artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)


def save_run_artifacts(run_name, y_true, y_pred, feature_cols, metadata):
    """Save local artifact files for one run and return the run artifact directory."""
    # TODO:
    # 1. create a run-specific artifact folder
    # 2. save confusion_matrix.png
    # 3. save classification_report.json
    # 4. save feature_columns.json
    # 5. save dataset_metadata_snapshot.json

    sanitized_name = "".join(
        "_" if not (char.isalnum() or char in "._-") else char
        for char in run_name
    )

    run_artifact_dir = ARTIFACT_DIR.joinpath(sanitized_name)
    run_artifact_dir.mkdir(parents=True, exist_ok=True)

    figure, axis = plt.subplots(figsize=(5, 4))
    ConfusionMatrixDisplay.from_predictions(
        y_true,
        y_pred,
        ax=axis,
        colorbar=False,
        values_format="d",
    )
    axis.set_title(run_name)
    figure.tight_layout()
    figure.savefig(run_artifact_dir / "confusion_matrix.png", dpi=150)
    plt.close(figure)

    report_path = run_artifact_dir / "classification_report.json"
    feature_path = run_artifact_dir / "feature_columns.json"
    metadata_path = run_artifact_dir / "dataset_metadata_snapshot.json"

    report = classification_report(
        y_true,
        y_pred,
        output_dict=True,
        zero_division=0,
    )

    with report_path.open("w", encoding="utf-8") as file:
        json.dump(report, file, indent=2)

    with feature_path.open("w", encoding="utf-8") as file:
        json.dump({"feature_columns": list(feature_cols)}, file, indent=2)

    with metadata_path.open("w", encoding="utf-8") as file:
        json.dump(metadata, file, indent=2, default=str)

    return run_artifact_dir


## 9. MLflow run helper

Complete a helper that:

1. fits the pipeline,
2. evaluates it,
3. logs params,
4. logs metrics,
5. logs tags,
6. logs artifacts,
7. logs the full sklearn Pipeline model.

Use the same helper for all model versions. That is the point of experiment tracking.

In [29]:
def run_mlflow_experiment(
    run_name,
    pipeline,
    X_train,
    X_test,
    y_train,
    y_test,
    feature_cols,
    model_params,
    tags,
    threshold=0.5,
):
    # TODO: implement this function.
    # Required MLflow calls:
    # - mlflow.start_run(run_name=run_name)
    # - mlflow.log_params(...)
    # - mlflow.log_metrics(...)
    # - mlflow.set_tags(...)
    # - mlflow.log_artifacts(...)
    # - mlflow.sklearn.log_model(...)

    pipeline.fit(X_train, y_train)

    metrics, predictions, scores = evaluate_binary_classifier(
        pipeline,
        X_test,
        y_test,
        threshold=threshold,
    )

    artifact_dir = save_run_artifacts(
        run_name,
        y_test,
        predictions,
        feature_cols,
        metadata,
    )

    logged_params = dict(model_params)
    logged_params.update(
        {
            "dataset_version": DATASET_VERSION,
            "feature_count": len(feature_cols),
            "threshold": threshold,
        }
    )

    logged_tags = dict(tags)
    logged_tags.update(
        {
            "homework": "hw02",
            "target": TARGET_COL,
        }
    )

    with mlflow.start_run(run_name=run_name) as active_run:
        mlflow.log_params(logged_params)
        mlflow.log_metrics(metrics)
        mlflow.set_tags(logged_tags)
        mlflow.log_artifacts(
            str(artifact_dir),
            artifact_path="evaluation_artifacts",
        )

        mlflow.sklearn.log_model(
            pipeline,
            artifact_path="model",
        )

        result = {
            "run_name": run_name,
            "run_id": active_run.info.run_id,
            **metrics,
        }

    print(result)
    return result

## 10. Run 0 — intentionally leaky model

This run is wrong on purpose.

Use a real model, but include `future_available_rate_30d`.

Expected behavior: performance may look suspiciously strong.

Required tags:

```text
leakage_status = leaky
known_defect = uses future_available_rate_30d
model_family = logistic_regression
```

In [31]:
# TODO:
# 1. split X_leaky and y using the same stratified split settings
# 2. build a LogisticRegression pipeline
# 3. log the run to MLflow

split_options = {
    "test_size": 0.20,
    "random_state": RANDOM_STATE,
    "stratify": y,
}

X_train_leaky, X_test_leaky, y_train_leaky, y_test_leaky = train_test_split(
    X_leaky,
    y,
    **split_options,
)

classifier = LogisticRegression(
    max_iter=1000,
    random_state=RANDOM_STATE,
)

leaky_logreg_pipeline = Pipeline(
    [
        ("preprocessor", preprocessor),
        ("classifier", classifier),
    ]
)

leaky_params = {
    "model_type": classifier.__class__.__name__,
    "max_iter": classifier.max_iter,
}

leaky_tags = {
    "uses_leakage": "true",
    "clean_candidate": "false",
}

v0_result = run_mlflow_experiment(
    run_name="v0_leaky_logistic_regression1",
    pipeline=leaky_logreg_pipeline,
    X_train=X_train_leaky,
    X_test=X_test_leaky,
    y_train=y_train_leaky,
    y_test=y_test_leaky,
    feature_cols=leaky_feature_cols,
    model_params=leaky_params,
    tags=leaky_tags,
)

2026/07/07 14:53:01 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run v0_leaky_logistic_regression1 at: http://185.50.38.163:33014/#/experiments/43/runs/1258d072735f4a938f178035e567ed03
🧪 View experiment at: http://185.50.38.163:33014/#/experiments/43
{'run_name': 'v0_leaky_logistic_regression1', 'run_id': '1258d072735f4a938f178035e567ed03', 'test_accuracy': 0.976145038167939, 'test_precision': 0.9760295021511985, 'test_recall': 0.9931207004377736, 'test_f1': 0.9845009299442034, 'test_roc_auc': 0.9892249054049123, 'prediction_threshold': 0.5}


## 11. Run 1 — dummy baseline

Train a `DummyClassifier(strategy="most_frequent")`.

This tells you what a useless model can achieve.

If your real model barely beats this, your model is weak.

In [32]:
# TODO: build and log dummy baseline.


baseline_classifier = DummyClassifier(
    strategy="most_frequent",
    random_state=RANDOM_STATE,
)

dummy_pipeline = Pipeline(
    [
        ("preprocessor", preprocessor),
        ("classifier", baseline_classifier),
    ]
)

dummy_params = {
    "model_type": baseline_classifier.__class__.__name__,
    "strategy": baseline_classifier.strategy,
}

dummy_tags = {
    "uses_leakage": "false",
    "clean_candidate": "true",
}

v1_result = run_mlflow_experiment(
    run_name="v1_dummy_baseline1",
    pipeline=dummy_pipeline,
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
    feature_cols=clean_feature_cols,
    model_params=dummy_params,
    tags=dummy_tags,
)

2026/07/07 14:53:17 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run v1_dummy_baseline1 at: http://185.50.38.163:33014/#/experiments/43/runs/2bd91cf22a794f2d9f982a2eec72edd5
🧪 View experiment at: http://185.50.38.163:33014/#/experiments/43
{'run_name': 'v1_dummy_baseline1', 'run_id': '2bd91cf22a794f2d9f982a2eec72edd5', 'test_accuracy': 0.762881679389313, 'test_precision': 0.762881679389313, 'test_recall': 1.0, 'test_f1': 0.8654939106901218, 'test_roc_auc': 0.5, 'prediction_threshold': 0.5}


## 12. Run 2 — clean logistic regression

Train your first clean real model.

Use only `X_clean`.

Required tags:

```text
leakage_status = clean
model_family = logistic_regression
```

In [33]:
# TODO: build and log clean LogisticRegression.

logistic_classifier = LogisticRegression(
    random_state=RANDOM_STATE,
    max_iter=1000,
)

clean_logreg_pipeline = Pipeline(
    [
        ("preprocessor", preprocessor),
        ("classifier", logistic_classifier),
    ]
)

clean_params = {
    "model_type": logistic_classifier.__class__.__name__,
    "max_iter": logistic_classifier.max_iter,
}

clean_tags = {
    "uses_leakage": "false",
    "clean_candidate": "true",
}

v2_result = run_mlflow_experiment(
    run_name="v2_clean_logistic_regression1",
    pipeline=clean_logreg_pipeline,
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
    feature_cols=clean_feature_cols,
    model_params=clean_params,
    tags=clean_tags,
)

2026/07/07 14:53:29 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run v2_clean_logistic_regression1 at: http://185.50.38.163:33014/#/experiments/43/runs/abd75929d5084f749cd900a9c580a1ff
🧪 View experiment at: http://185.50.38.163:33014/#/experiments/43
{'run_name': 'v2_clean_logistic_regression1', 'run_id': 'abd75929d5084f749cd900a9c580a1ff', 'test_accuracy': 0.976145038167939, 'test_precision': 0.9760295021511985, 'test_recall': 0.9931207004377736, 'test_f1': 0.9845009299442034, 'test_roc_auc': 0.9892249054049123, 'prediction_threshold': 0.5}


## 13. Run 3 — class-weighted logistic regression

Train logistic regression with:

```python
class_weight="balanced"
```

Compare precision and recall against the previous clean logistic model.

In [34]:
# TODO: build and log class-weighted LogisticRegression.

balanced_classifier = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=RANDOM_STATE,
)

balanced_logreg_pipeline = Pipeline(
    [
        ("preprocessor", preprocessor),
        ("classifier", balanced_classifier),
    ]
)

balanced_params = {
    "model_type": balanced_classifier.__class__.__name__,
    "max_iter": balanced_classifier.max_iter,
    "class_weight": balanced_classifier.class_weight,
}

balanced_tags = {
    "uses_leakage": "false",
    "clean_candidate": "true",
}

v3_result = run_mlflow_experiment(
    run_name="v3_balanced_logistic_regression1",
    pipeline=balanced_logreg_pipeline,
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
    feature_cols=clean_feature_cols,
    model_params=balanced_params,
    tags=balanced_tags,
)

2026/07/07 14:53:41 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run v3_balanced_logistic_regression1 at: http://185.50.38.163:33014/#/experiments/43/runs/88f9106ff1964ed394877999e510e0c8
🧪 View experiment at: http://185.50.38.163:33014/#/experiments/43
{'run_name': 'v3_balanced_logistic_regression1', 'run_id': '88f9106ff1964ed394877999e510e0c8', 'test_accuracy': 0.9770992366412213, 'test_precision': 0.9813780260707635, 'test_recall': 0.9887429643527205, 'test_f1': 0.9850467289719627, 'test_roc_auc': 0.9905587370376102, 'prediction_threshold': 0.5}


## 14. Run 4 — threshold tuning

Use a fitted probability model and test several decision thresholds.

Suggested thresholds:

```text
0.30, 0.40, 0.50, 0.60
```

You may log one run per threshold.

The goal is to see how precision/recall/f1 change when the threshold changes.

In [35]:
# TODO: log threshold-tuning runs.

threshold_values = [0.30, 0.40, 0.50, 0.60, 0.70]
threshold_results = []

base_classifier = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=RANDOM_STATE,
)

for t in threshold_values:
    pipeline = Pipeline(
        [
            ("preprocessor", preprocessor),
            ("classifier", base_classifier),
        ]
    )

    result = run_mlflow_experiment(
        run_name=f"v4_balanced_logistic_threshold1_{t:.2f}",
        pipeline=pipeline,
        X_train=X_train,
        X_test=X_test,
        y_train=y_train,
        y_test=y_test,
        feature_cols=clean_feature_cols,
        model_params={
            "model_type": base_classifier.__class__.__name__,
            "max_iter": base_classifier.max_iter,
            "class_weight": base_classifier.class_weight,
            "threshold_tuning": True,
        },
        tags={
            "uses_leakage": "false",
            "clean_candidate": "true",
            "threshold_tuning": "true",
        },
        threshold=t,
    )

    threshold_results.append(result)

threshold_results_df = pd.DataFrame(threshold_results)
threshold_results_df.sort_values("test_f1", ascending=False)

2026/07/07 14:54:12 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run v4_balanced_logistic_threshold1_0.30 at: http://185.50.38.163:33014/#/experiments/43/runs/f987bcc41fe34245bef00165ddbbecbb
🧪 View experiment at: http://185.50.38.163:33014/#/experiments/43
{'run_name': 'v4_balanced_logistic_threshold1_0.30', 'run_id': 'f987bcc41fe34245bef00165ddbbecbb', 'test_accuracy': 0.9770992366412213, 'test_precision': 0.977818853974122, 'test_recall': 0.9924953095684803, 'test_f1': 0.9851024208566108, 'test_roc_auc': 0.9905587370376102, 'prediction_threshold': 0.3}


2026/07/07 14:54:24 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run v4_balanced_logistic_threshold1_0.40 at: http://185.50.38.163:33014/#/experiments/43/runs/25484ef841e8475fb07082f384640643
🧪 View experiment at: http://185.50.38.163:33014/#/experiments/43
{'run_name': 'v4_balanced_logistic_threshold1_0.40', 'run_id': '25484ef841e8475fb07082f384640643', 'test_accuracy': 0.9780534351145038, 'test_precision': 0.9802102659245516, 'test_recall': 0.9912445278298937, 'test_f1': 0.9856965174129353, 'test_roc_auc': 0.9905587370376102, 'prediction_threshold': 0.4}


2026/07/07 14:54:36 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


...

2026/07/07 14:54:47 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run v4_balanced_logistic_threshold1_0.60 at: http://185.50.38.163:33014/#/experiments/43/runs/aaeea29d72364e72823cf0c684c2a05a
🧪 View experiment at: http://185.50.38.163:33014/#/experiments/43
{'run_name': 'v4_balanced_logistic_threshold1_0.60', 'run_id': 'aaeea29d72364e72823cf0c684c2a05a', 'test_accuracy': 0.9770992366412213, 'test_precision': 0.9825762289981331, 'test_recall': 0.9874921826141339, 'test_f1': 0.9850280723643169, 'test_roc_auc': 0.9905587370376102, 'prediction_threshold': 0.6}


2026/07/07 14:54:58 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run v4_balanced_logistic_threshold1_0.70 at: http://185.50.38.163:33014/#/experiments/43/runs/af2b629cc23441e396dcd4155c3a99fd
🧪 View experiment at: http://185.50.38.163:33014/#/experiments/43
{'run_name': 'v4_balanced_logistic_threshold1_0.70', 'run_id': 'af2b629cc23441e396dcd4155c3a99fd', 'test_accuracy': 0.9790076335877863, 'test_precision': 0.9850280723643169, 'test_recall': 0.9874921826141339, 'test_f1': 0.9862585883822611, 'test_roc_auc': 0.9905587370376102, 'prediction_threshold': 0.7}


,run_name,run_id,test_accuracy,test_precision,test_recall,test_f1,test_roc_auc,prediction_threshold
4,v4_balanced_logistic_threshold1_0.70,af2b629cc23441e396dcd4155c3a99fd,0.979008,0.985028,0.987492,0.986259,0.990559,0.7
1,v4_balanced_logistic_threshold1_0.40,25484ef841e8475fb07082f384640643,0.978053,0.980210,0.991245,0.985697,0.990559,0.4
0,v4_balanced_logistic_threshold1_0.30,f987bcc41fe34245bef00165ddbbecbb,0.977099,0.977819,0.992495,0.985102,0.990559,0.3
2,v4_balanced_logistic_threshold1_0.50,0761677bfee54da0ac98c5205045951e,0.977099,0.981378,0.988743,0.985047,0.990559,0.5
3,v4_balanced_logistic_threshold1_0.60,aaeea29d72364e72823cf0c684c2a05a,0.977099,0.982576,0.987492,0.985028,0.990559,0.6


## 15. Run 5 — tree-based model

Train a `RandomForestClassifier`.

This compares a nonlinear model against logistic regression.

Log at least these parameters:

```text
n_estimators
max_depth
min_samples_leaf
class_weight
random_state
```

In [37]:
# TODO: build and log RandomForestClassifier.

rf_classifier = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    min_samples_leaf=2,
    class_weight="balanced_subsample",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

random_forest_pipeline = Pipeline(
    [
        ("preprocessor", preprocessor),
        ("classifier", rf_classifier),
    ]
)

rf_params = {
    "model_type": rf_classifier.__class__.__name__,
    "n_estimators": rf_classifier.n_estimators,
    "min_samples_leaf": rf_classifier.min_samples_leaf,
    "class_weight": rf_classifier.class_weight,
}

rf_tags = {
    "uses_leakage": "false",
    "clean_candidate": "true",
}

v5_result = run_mlflow_experiment(
    run_name="v5_random_forest1",
    pipeline=random_forest_pipeline,
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
    feature_cols=clean_feature_cols,
    model_params=rf_params,
    tags=rf_tags,
)

2026/07/07 14:56:52 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run v5_random_forest1 at: http://185.50.38.163:33014/#/experiments/43/runs/1cde530ae5b54ef7b05e077b3aa14cf0
🧪 View experiment at: http://185.50.38.163:33014/#/experiments/43
{'run_name': 'v5_random_forest1', 'run_id': '1cde530ae5b54ef7b05e077b3aa14cf0', 'test_accuracy': 0.9837786259541985, 'test_precision': 0.9869321717485999, 'test_recall': 0.991869918699187, 'test_f1': 0.9893948845913911, 'test_roc_auc': 0.9926890926547401, 'prediction_threshold': 0.5}


## 16. Compare MLflow runs

Use `mlflow.search_runs` to retrieve your experiment runs.

Compare at least:

```text
run name
leakage status
model family
accuracy
precision
recall
f1
roc_auc
```

Do not select a leaky run as final candidate.

In [38]:
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

# TODO: retrieve MLflow runs for this experiment and create a comparison table.

if experiment is None:
    raise ValueError(f"Experiment does not exist: {EXPERIMENT_NAME}")

runs_df = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.test_f1 DESC"],
)

base_columns = [
    "run_id",
    "tags.mlflow.runName",
    "tags.uses_leakage",
    "tags.clean_candidate",
    "metrics.test_accuracy",
    "metrics.test_precision",
    "metrics.test_recall",
    "metrics.test_f1",
    "metrics.test_roc_auc",
    "metrics.prediction_threshold",
    "params.model_type",
    "params.feature_count",
]

available_columns = list(filter(lambda c: c in runs_df.columns, base_columns))

comparison_df = runs_df.loc[:, available_columns].copy()

if "metrics.test_f1" in comparison_df.columns:
    comparison_df = comparison_df.sort_values("metrics.test_f1", ascending=False)
comparison_df

,run_id,tags.mlflow.runName,tags.uses_leakage,tags.clean_candidate,metrics.test_accuracy,metrics.test_precision,metrics.test_recall,metrics.test_f1,metrics.test_roc_auc,metrics.prediction_threshold,params.model_type,params.feature_count
0,1cde530ae5b54ef7b05e077b3aa14cf0,v5_random_forest1,false,true,0.983779,0.986932,0.991870,0.989395,0.992689,0.5,RandomForestClassifier,26
1,9723c887ac5b4cb6bdd01b252b70157d,v5_random_forest1,false,true,0.983779,0.986932,0.991870,0.989395,0.992689,0.5,RandomForestClassifier,26
2,28f586690eb0421e93e7eaba3f38d353,v5_random_forest,false,true,0.983302,0.986318,0.991870,0.989086,0.994519,0.5,RandomForestClassifier,24
3,59c4aad277644990a88093ecfa4e7191,v5_random_forest,false,true,0.983302,0.986318,0.991870,0.989086,0.994519,0.5,RandomForestClassifier,24
4,af2b629cc23441e396dcd4155c3a99fd,v4_balanced_logistic_threshold1_0.70,false,true,0.979008,0.985028,0.987492,0.986259,0.990559,0.7,LogisticRegression,26
5,3bb7194a4e0a4b67b269ca9f4e9a2bc4,v4_balanced_logistic_threshold_0.70,false,true,0.979008,0.985028,0.987492,0.986259,0.990595,0.7,LogisticRegression,24
6,2ef41e9cdd9947cf8935e13b8ee74c32,v4_balanced_logistic_threshold_0.70,false,true,0.979008,0.985028,0.987492,0.986259,0.990595,0.7,LogisticRegression,24
7,25484ef841e8475fb07082f384640643,v4_balanced_logistic_threshold1_0.40,false,true,0.978053,0.980210,0.991245,0.985697,0.990559,0.4,LogisticRegression,26
8,f9115aaae98e471790fe54f93c3babe0,v4_balanced_logistic_threshold_0.60,false,true,0.977576,0.983188,0.987492,0.985335,0.990595,0.6,LogisticRegression,24
9,6d2c978d95464a04942a6c1d5a2821a4,v4_balanced_logistic_threshold_0.60,false,true,0.977576,0.983188,0.987492,0.985335,0.990595,0.6,LogisticRegression,24


## 17. Select final candidate

Pick the best **clean** run.

Do not choose the leaky run.

Selection should be based on:

- f1
- roc_auc
- precision/recall tradeoff
- no leakage
- full preprocessing Pipeline logged

Write a short explanation.

In [39]:
# TODO: set BEST_RUN_ID to the selected clean run ID.

clean_only_runs = comparison_df.copy()

if "tags.uses_leakage" in clean_only_runs.columns:
    clean_only_runs = clean_only_runs[clean_only_runs["tags.uses_leakage"] != "true"]

if "tags.clean_candidate" in clean_only_runs.columns:
    clean_only_runs = clean_only_runs[clean_only_runs["tags.clean_candidate"] == "true"]

if len(clean_only_runs) == 0:
    raise ValueError("No clean candidate runs found. Run the clean experiments first.")

best_row = clean_only_runs.sort_values(
    "metrics.test_f1", ascending=False
).iloc[0]

BEST_RUN_ID = best_row["run_id"]
if BEST_RUN_ID is None:
    raise ValueError("Set BEST_RUN_ID to your selected clean MLflow run ID.")

client.set_tag(BEST_RUN_ID, "selected_for_serving", "true")
client.set_tag(BEST_RUN_ID, "production_candidate", "true")

print("Selected best run:", BEST_RUN_ID)


Selected best run: 1cde530ae5b54ef7b05e077b3aa14cf0


## Final explanation

Write 3–6 sentences:

- Which run did you select?
- Why did you select it?
- Why did you reject the leaky run?
- What would you try next?

In [40]:
# TODO: replace this text.
final_explanation = f"""
This project evaluated multiple machine learning models under a controlled MLflow experiment setup to predict high demand behavior in Airbnb listings.

A baseline and several improved models were trained, including Logistic Regression, class-weighted Logistic Regression, and Random Forest. An additional experiment was performed to demonstrate the impact of data leakage using a future-derived feature.

Models were compared using standard classification metrics, and results were tracked in MLflow for reproducibility and analysis. The final selection was made based on the highest F1-score among clean (non-leaky) runs.

The selected model for serving is {BEST_RUN_ID}. This run represents the best trade-off between precision and recall on the clean dataset and is therefore chosen as the production candidate.

For deployment, only the approved feature set (excluding leakage-related and future-window columns) should be used to ensure consistency between training and inference.
"""

print(final_explanation)



This project evaluated multiple machine learning models under a controlled MLflow experiment setup to predict high demand behavior in Airbnb listings.

A baseline and several improved models were trained, including Logistic Regression, class-weighted Logistic Regression, and Random Forest. An additional experiment was performed to demonstrate the impact of data leakage using a future-derived feature.

Models were compared using standard classification metrics, and results were tracked in MLflow for reproducibility and analysis. The final selection was made based on the highest F1-score among clean (non-leaky) runs.

The selected model for serving is 1cde530ae5b54ef7b05e077b3aa14cf0. This run represents the best trade-off between precision and recall on the clean dataset and is therefore chosen as the production candidate.

For deployment, only the approved feature set (excluding leakage-related and future-window columns) should be used to ensure consistency between training and infere